# FINRL Walk-Forward Experiment

This notebook runs the Phase 12 walk-forward experiment runner and visualizes portfolio performance against the S&P 500 / SPY benchmark, plus spectral feature evolution over time.

Use small synthetic data locally. Use Colab for the full configured universe, encoder training, PPO training, and full walk-forward experiments.

In [35]:
# Colab setup. Uncomment after cloning the repository in Colab.
# %pip install -e .

from datetime import date, timedelta

import polars as pl

from finrl.backtest.walk_forward import WalkForwardConfig
from finrl.data import (
    MarketDataBundle,
    MarketDataConfig,
    UniverseConfig,
    build_weekly_rebalance_calendar,
    compute_open_to_open_returns,
    download_ohlcv,
)
from finrl.data.download import download_macro_series
from finrl.experiments import (
    ExperimentConfig,
    RawExperimentData,
    build_allocation_figure,
    build_performance_figure,
    build_spectral_figure,
    metrics_to_frame,
    run_walk_forward_experiment,
)
from finrl.features import FeatureConfig, build_feature_bundle
from finrl.features.preprocessing import PreprocessingConfig
from finrl.features.schema import FeatureBundle
from finrl.models.encoder import EncoderConfig
from finrl.ppo.policy import PPOConfig
from finrl.regimes.schema import HMMConfig

## Prepared Data Contract

The runner expects prepared feature and return tables:

- `FeatureBundle` with asset, macro, and 20 spectral feature columns.
- `returns`: Polars DataFrame with `decision_date` and one return column per tradable asset, including cash.
- `spy_returns`: Polars DataFrame with `decision_date` and `spy_return` for the same holding periods.

Replace the synthetic fixture below with the output of the data, feature, preprocessing, and return-preparation pipeline for full experiments.

## Run With Real yfinance Data

Edit `TICKERS`, `START`, `END`, and `MAX_STOCKS`, then run this section in Colab. The code downloads real stock data plus SPY, computes daily open-to-open returns, builds causal features, adds a CASH asset, and packages everything into `RawExperimentData` for the walk-forward runner.

In [36]:
TICKERS = [
    "SPY",
    "VGT",
    "NVDA",
    "IAU"
]
MAX_STOCKS = len(TICKERS)  # set to 100 after pasting your full universe
START = "2000-01-01"
END = "2026-06-05"
CACHE_DIR = "data/cache"
BENCHMARK_TICKER = "SPY"
CASH_RETURN_PER_PERIOD = 0.0

universe = UniverseConfig(
    tickers=TICKERS,
    max_stocks=MAX_STOCKS,
    include_cash=True,
    cash_ticker="CASH",
    benchmark_ticker=BENCHMARK_TICKER,
)
market_config = MarketDataConfig(
    universe=universe,
    start=START,
    end=END,
    cache_dir=CACHE_DIR,
)
selected_tickers = universe.selected_tickers
selected_tickers

('SPY', 'VGT', 'NVDA', 'IAU')

In [37]:
def _returns_wide(open_to_open_returns: pl.DataFrame, tickers: tuple[str, ...], cash_return: float) -> pl.DataFrame:
    wide = (
        open_to_open_returns
        .select(["decision_date", "ticker", "return"])
        .pivot(index="decision_date", on="ticker", values="return", aggregate_function="first")
        .sort("decision_date")
    )
    return (
        wide
        .select(["decision_date", *tickers])
        .with_columns(pl.lit(cash_return).alias("CASH"))
        .drop_nulls()
    )


def _spy_returns(open_to_open_returns: pl.DataFrame) -> pl.DataFrame:
    return (
        open_to_open_returns
        .select(["decision_date", pl.col("return").alias("spy_return")])
        .sort("decision_date")
        .drop_nulls()
    )


def _filter_features_to_common_dates(features: FeatureBundle, returns: pl.DataFrame, spy_returns: pl.DataFrame) -> FeatureBundle:
    common_dates = (
        returns.select("decision_date")
        .join(spy_returns.select("decision_date"), on="decision_date", how="inner")
        .rename({"decision_date": "date"})
        .with_columns(pl.col("date").cast(pl.Date))
        .unique()
        .sort("date")
    )
    asset = features.asset_features.join(common_dates, on="date", how="inner").sort(["date", "ticker"])
    macro = (
        common_dates
        .join(features.macro_features, on="date", how="left")
        .sort("date")
        .with_columns(pl.all().exclude("date").forward_fill().fill_null(0.0))
    )
    spectral = features.spectral_features.join(common_dates, on="date", how="inner").sort("date")
    dates = tuple(common_dates.get_column("date").to_list())
    return FeatureBundle(
        asset_features=asset,
        macro_features=macro,
        spectral_features=spectral,
        decision_dates=dates,
        tickers=features.tickers,
        asset_feature_columns=features.asset_feature_columns,
        macro_feature_columns=features.macro_feature_columns,
        spectral_feature_columns=features.spectral_feature_columns,
    )


def make_real_yfinance_data() -> RawExperimentData:
    ohlcv = download_ohlcv(selected_tickers, START, END, market_config)
    spy_ohlcv = download_ohlcv((BENCHMARK_TICKER,), START, END, market_config)
    macro = download_macro_series(START, END, market_config)
    calendar = build_weekly_rebalance_calendar(ohlcv)

    market_bundle = MarketDataBundle(
        ohlcv=ohlcv,
        spy_ohlcv=spy_ohlcv,
        macro=macro,
        calendar=calendar,
    )
    features = build_feature_bundle(
        market_bundle,
        FeatureConfig(spectral_dim=20, include_hawkes=False),
    )

    stock_returns = _returns_wide(
        compute_open_to_open_returns(ohlcv, calendar),
        selected_tickers,
        CASH_RETURN_PER_PERIOD,
    )
    spy_returns = _spy_returns(compute_open_to_open_returns(spy_ohlcv, calendar))
    features = _filter_features_to_common_dates(features, stock_returns, spy_returns)
    common_dates = pl.DataFrame({"decision_date": list(features.decision_dates)}).with_columns(pl.col("decision_date").cast(pl.Date))
    stock_returns = common_dates.join(stock_returns, on="decision_date", how="inner")
    spy_returns = common_dates.join(spy_returns, on="decision_date", how="inner")
    return RawExperimentData(features=features, returns=stock_returns, spy_returns=spy_returns)


raw_data = make_real_yfinance_data()
raw_data.features.asset_features.head(), raw_data.returns.head(), raw_data.spy_returns.head()

(shape: (5, 22)
 ┌────────────┬────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
 │ date       ┆ ticker ┆ open       ┆ high      ┆ … ┆ macd_hist ┆ return_pe ┆ dollar_vo ┆ amihud_pe │
 │ ---        ┆ ---    ┆ ---        ┆ ---       ┆   ┆ ---       ┆ rcentile_ ┆ lume_perc ┆ rcentile_ │
 │ date       ┆ str    ┆ f64        ┆ f64       ┆   ┆ f64       ┆ rank      ┆ entile_ra ┆ rank      │
 │            ┆        ┆            ┆           ┆   ┆           ┆ ---       ┆ nk        ┆ ---       │
 │            ┆        ┆            ┆           ┆   ┆           ┆ f64       ┆ ---       ┆ f64       │
 │            ┆        ┆            ┆           ┆   ┆           ┆           ┆ f64       ┆           │
 ╞════════════╪════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
 │ 2005-01-28 ┆ IAU    ┆ 8.546      ┆ 8.546     ┆ … ┆ 0.0       ┆ null      ┆ 0.333333  ┆ null      │
 │ 2005-01-28 ┆ NVDA   ┆ 0.191667   ┆ 0.192417  ┆ … ┆ 0.000342  ┆ 

In [38]:
n_stocks = len(raw_data.features.tickers)
n_tradable_assets = len([column for column in raw_data.returns.columns if column != "decision_date"])
asset_feature_dim = len(raw_data.features.asset_feature_columns)
macro_feature_dim = len(raw_data.features.macro_feature_columns)

config = ExperimentConfig(
    walk_forward=WalkForwardConfig(train_years=2, test_years=1, step_years=1),
    preprocessing=PreprocessingConfig(rolling_window=252),
    encoder=EncoderConfig(
        lookback=60,
        n_assets=n_stocks,
        asset_feature_dim=asset_feature_dim,
        macro_feature_dim=macro_feature_dim,
        spectral_feature_dim=20,
    ),
    hmm=HMMConfig(n_states=4, max_iter=50),
    ppo=PPOConfig(n_assets=n_tradable_assets, train_epochs=100, learning_rate=1e-4),
    enable_ppo=True,
    seed=7,
    periods_per_year=252,
)

print({
    "stocks": n_stocks,
    "tradable_assets_including_cash": n_tradable_assets,
    "asset_feature_dim": asset_feature_dim,
    "macro_feature_dim": macro_feature_dim,
    "decision_dates": len(raw_data.features.decision_dates),
})

result = run_walk_forward_experiment(raw_data, config)
metrics_to_frame(result)

{'stocks': 4, 'tradable_assets_including_cash': 5, 'asset_feature_dim': 20, 'macro_feature_dim': 24, 'decision_dates': 1077}


Model is not converging.  Current: 625.2106300223411 is not greater than 625.2108691759419. Delta is -0.00023915360077353398
Model is not converging.  Current: 628.0312039371348 is not greater than 628.031203969676. Delta is -3.2541151995246764e-08
Model is not converging.  Current: 553.7790440865168 is not greater than 553.7790444964631. Delta is -4.09946323998156e-07
Model is not converging.  Current: 238.0788627997039 is not greater than 238.07900557586942. Delta is -0.0001427761655179438


split_index,test_start,test_end,portfolio_cumulative_return,spy_cumulative_return,spy_relative_alpha,portfolio_max_drawdown,portfolio_mean_turnover,portfolio_total_transaction_cost
i64,date,date,f64,f64,f64,f64,f64,f64
0,2007-01-01,2007-12-31,0.170474,0.00703,0.163444,0.083948,0.236218,0.012047
1,2008-01-01,2008-12-31,-0.342052,-0.346802,0.00475,0.411633,0.256341,0.012817
2,2009-01-01,2009-12-31,0.396533,0.242362,0.15417,0.093959,0.246988,0.012102
3,2010-01-01,2010-12-31,0.115348,0.09993,0.015418,0.122753,0.210503,0.010525
4,2011-01-01,2011-12-31,-0.014538,0.011218,-0.025756,0.171331,0.253055,0.012906
…,…,…,…,…,…,…,…,…
15,2022-01-01,2022-12-31,-0.157022,-0.156322,-0.000701,0.264388,0.227508,0.011603
16,2023-01-01,2023-12-31,0.366345,0.199964,0.166381,0.056708,0.222916,0.011369
17,2024-01-01,2024-12-31,0.507067,0.272912,0.234155,0.12489,0.225152,0.011483


## Portfolio Allocation


In [39]:
result.allocations.head()


decision_date,split_index,SPY,VGT,NVDA,IAU,CASH
date,i64,f32,f32,f32,f32,f32
2007-01-05,0,0.15281,0.192788,0.262594,0.235363,0.156445
2007-01-12,0,0.246485,0.137758,0.299901,0.134914,0.180942
2007-01-19,0,0.187361,0.225837,0.224835,0.239294,0.122672
2007-01-26,0,0.268486,0.160665,0.175322,0.141886,0.253641
2007-02-02,0,0.174095,0.212805,0.188675,0.22514,0.199285


In [40]:
allocation_fig = build_allocation_figure(result, top_n=5)
allocation_fig.show()


## Performance vs S&P 500

In [41]:
performance_fig = build_performance_figure(result)
performance_fig.show()

## Spectral Feature Evolution

In [42]:
spectral_fig = build_spectral_figure(result, value_columns=("volume_eigen_0", "volume_eigen_1", "volume_eigen_2", "volume_eigen_3"))
spectral_fig.show()

In [43]:
# Optional report export
from finrl.experiments import write_report
write_report(result, "walk_forward_report")